In [ ]:
import sys
from pathlib import Path

path = Path().cwd().parent / "src"
sys.path.insert(0, str(path))

In [ ]:
from dask_obj.expr import *
from dask_obj.core import *

In [ ]:
from itables import init_notebook_mode, show

init_notebook_mode(all_interactive=True)

In [ ]:
import itables.options as opt

opt.maxBytes = 0
opt.maxColumns = 1000
opt.maxRows = 1000

In [ ]:
import gzip
import math
from collections import Counter
from copy import deepcopy
from datetime import datetime
from functools import total_ordering
from io import StringIO
from itertools import islice
from operator import attrgetter, itemgetter, methodcaller
from pathlib import Path
from typing import Iterable

from dask import compute, persist
from dask.delayed import delayed
from dask.distributed import Client, LocalCluster, get_client
from IPython.display import display
import dask
import dask.array as da
import dask.bag as db
import dask.dataframe as dd
import fsspec
import glom
import pandas as pd
import polars as pl
import toolz

In [ ]:
pl.Config.set_tbl_cols(1000)

s = pl.Series("a", [1, 2, 3, 4, 5])
print(s)

df = pl.DataFrame(
    {
        "integer": [1, 2, 3, 4, 5],
        "date": [
            datetime(2022, 1, 1),
            datetime(2022, 1, 2),
            datetime(2022, 1, 3),
            datetime(2022, 1, 4),
            datetime(2022, 1, 5),
        ],
        "float": [4.0, 5.0, 6.0, 7.0, 8.0],
    }
)

print(df)

df2 = pl.DataFrame(
    {
        "integer": range(6, 11),
        "date": [
            datetime(2022, 1, 6),
            datetime(2022, 1, 7),
            datetime(2022, 1, 8),
            datetime(2022, 1, 9),
            datetime(2022, 1, 10),
        ],
        "float": [9.0, 10.0, 11.0, 12.0, 13.0],
    }
)

print(df2)

objs = DaskObjects((df, df2))
objs

objs.head(3).compute()

objs.describe().compute()

objs.map(lambda df: df.filter((pl.col("integer") >= 5) & (pl.col("integer") <= 6))).compute()

result = objs.filter((pl.col("integer") >= 5) & (pl.col("integer") <= 6)).compute()
result

pl.concat(result)

In [ ]:
path = "~/e/data/gharchive"
path = Path(path).expanduser()
print(path)
files = list(map(str, [*path.glob("*.json.gz")]))
files[:5]

In [ ]:
file = "/home/brl0/e/data/gharchive/2023-03-01-11.json/2023-03-01-11.json"

df = pl.read_ndjson(file)
df.head(2)

df_pd = pd.read_json(file, orient="records", lines=True, dtype_backend="pyarrow")
df_pd.head(2)

df_pd = df_pd.convert_dtypes()
df_pd.head(2)

df_pd.payload.sample(1).iloc[0]

def get_forkee_owner_login(data):
    try:
        return glom.glom(data, "forkee.owner.login")
    except glom.core.PathAccessError:
        return None

forkee_owner_login = df_pd.payload.apply(get_forkee_owner_login)

forkee_owner_login.value_counts()

actor = df_pd.actor.apply(pd.Series).convert_dtypes()
actor

repo = df_pd.repo.apply(pd.Series).convert_dtypes()
repo

org = df_pd.org.apply(pd.Series).convert_dtypes()
org

payload = df_pd.payload.apply(pd.Series).convert_dtypes()
payload

payload.apply(lambda x: x.apply(type).value_counts()).convert_dtypes()

issue = payload.issue.apply(pd.Series).convert_dtypes()
issue

issue.apply(lambda _: _.apply(type).value_counts()).convert_dtypes().T

In [ ]:
# def pl_read_ndjson_fsspec(file):
#     with fsspec.open(file, "rt", compression="infer") as f:
#         data = f.read()
#     data = StringIO(data)
#     return pl.read_ndjson(data)

In [ ]:
def pl_read_ndjson_fsspec(file, lines=None):
    with fsspec.open(file, "rb", compression="infer") as f:
        if lines is not None:
            data = b"".join(islice(f, lines))
        else:
            data = f.read()
    return pl.read_ndjson(data)

In [ ]:
def unnest(df):
    """Unnest a dataframe with struct columns."""
    data = []
    columns = []
    for col, dtype in zip(df.columns, df.dtypes):
        if dtype == pl.Struct:
            fields = df[col].struct.fields
            _rename_fields = [f"{col}_{f}" for f in fields]
            rename_fields = []
            for f in _rename_fields:
                if f in columns:
                    i = 1
                    while f"{f}_{i}" in columns:
                        i += 1
                    f = f"{f}_{i}"
                rename_fields.append(f)
            unnested = (
                df[col]
                .struct.rename_fields(rename_fields)
                .to_frame()
                .unnest(col)
                .pipe(unnest)
            )
            data.extend(unnested.get_columns())
            columns.extend(unnested.columns)
        else:
            data.append(df[col])
            columns.append(col)
    out = pl.DataFrame(data)
    return out

def explode(df):
    """Explode a dataframe with list columns."""
    for col, dtype in zip(df.columns, df.dtypes):
        if dtype == pl.List:
            df = df.explode(col)
    return df

def expand(df):
    """Expand a dataframe with struct and list columns."""
    while any(_ == pl.Struct or _ == pl.List for _ in df.dtypes):
        df = df.pipe(explode).pipe(unnest)
    return df

In [ ]:
# df = pl.read_ndjson(file)

In [ ]:
# df = pl_read_ndjson_fsspec(files[3], 1991)
# df

In [ ]:
# df = pl_read_ndjson_fsspec(files[0], 1000)
# df

In [ ]:
cluster = LocalCluster(n_workers=10, memory_limit=None)
client = Client(cluster)

In [ ]:
# def process(file):
#     return pl_read_ndjson_fsspec(file, lines=1000).pipe(unnest).to_pandas(use_pyarrow_extension_array=True)

In [ ]:
lines = None

def process(file):
    path = Path(file)
    name = path.name.split(".")[0]
    path = path.parent / f"parquet/{name}.parquet"
    if not path.exists():
        pl_read_ndjson_fsspec(file, lines=lines).pipe(unnest).write_parquet(path)
    return str(path)

In [ ]:
objs = DaskDelayedObjects(files[:5])
dfs = objs.map(process).compute()
del objs

In [ ]:
dfs

In [ ]:
ddf = dd.read_parquet(dfs)
ddf

In [ ]:
kwargs = dict(enforce_metadata=False, transform_divisions=False, align_dataframes=False)
ddf.map_partitions(lambda df: tuple(df.columns), **kwargs).compute()

In [ ]:
dfs[0]

In [ ]:
df = pl.scan_parquet("/home/brl0/e/data/gharchive/parquet/*.parquet").collect()

In [ ]:
client.close()
cluster.close()

In [ ]:
df = pl.from_pandas(pd.concat(dfs, ignore_index=True))
del dfs
print(df.shape)
df.head(3)

In [ ]:
objs = DaskDelayedObjects(files[:4])
dfs = objs.map(pl_read_ndjson_fsspec, lines=1000)  #.persist()

In [ ]:
unnested = dfs.map(unnest)  #.persist()

In [ ]:
pd_dfs = unnested.to_pandas()  #.persist()

In [ ]:
pd_dfs

In [ ]:
results = pd_dfs.compute()

In [ ]:
del objs, dfs, unnested, pd_dfs

In [ ]:
client.close()
cluster.close()

In [ ]:
# df = pl.concat(results, how="diagonal")
# df = pd.concat(results)
# df = pl.from_pandas(df)
# df

In [ ]:
df = pl.from_pandas(pd.concat(results))
del results

In [ ]:
df.shape

In [ ]:
df.head(3)

In [ ]:
df.estimated_size() / 1024 ** 3

In [ ]:
dfs = [pl_read_ndjson_fsspec(f).pipe(unnest) for f in files[3:5]]
df = pl.concat(dfs, how="diagonal")
df

In [ ]:
df = pl.concat(dfs, how="diagonal")
df

In [ ]:
url = "https://data.gharchive.org/{}-{:02d}-{:02d}-{}.json.gz"
dates = ((2023, 3, 1, 1), (2023, 3, 1, 10))
dfs = [pl_read_ndjson_fsspec(url.format(*d)) for d in dates]
df = pl.concat(dfs)
df

In [ ]:
url = "https://data.gharchive.org/{}-{:02d}-{:02d}-{}.json.gz"
dates = ((2023, 3, 1, 1), (2023, 3, 1, 10))
dfs = [pl_read_ndjson_fsspec(url.format(*d)).pipe(unnest) for d in dates]
df = pl.concat(dfs)
df

In [ ]:
pl.show_versions()

In [ ]:
import pyarrow as pa

In [ ]:
df0 = dfs[0].to_pandas(use_pyarrow_extension_array=True)
df1 = dfs[1].to_pandas(use_pyarrow_extension_array=True)

df = pd.concat([df0, df1], axis=1)
df

In [ ]:
from io import StringIO
import fsspec
import polars as pl

def pl_read_ndjson_fsspec(file, lines=None):
    with fsspec.open(file, "rt", compression="infer") as f:
        data = f.read()
    data = StringIO(data)
    return pl.read_ndjson(data)

url = "https://data.gharchive.org/{}-{:02d}-{:02d}-{}.json.gz"
dates = ((2023, 3, 1, 1), (2023, 3, 1, 10))
dfs = [pl_read_ndjson_fsspec(url.format(*d)) for d in dates]
df = pl.concat(dfs)
df

In [ ]:
import pandas as pd

df0 = dfs[0].pipe(unnest).to_pandas()
df1 = dfs[1].pipe(unnest).to_pandas()

df = pd.concat([df0, df1])
df

In [ ]:
df = pl.from_pandas(df)
df

In [ ]:
df = unnest(df)
df